In [2]:
pip install transformers peft trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install bitsandbytes>=0.46.1

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/input/datasets/ibtihussain/aicr-sft-adapter-v3-nosysprmpt" #fine tuned without sys prompts - qwen model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# IMPORTANT: is_trainable=True — unlike your earlier inference-only loads,
# DPO needs to continue updating this adapter's weights
dpo_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH, is_trainable=True)

print("Active adapters:", dpo_model.active_adapters)
print("Trainable params:")
dpo_model.print_trainable_parameters()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Active adapters: ['default']
Trainable params:
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [2]:
import json
import random

PREFS_PATH = "/kaggle/input/datasets/ibtihussain/pref-pair-sft-dpo/preference_pairs.jsonl"
DPO_TRAIN_PATH = "/kaggle/working/dpo_train_nosysPrmpt.jsonl"

SEED = 10  # SAME seed as SFT split, so train/holdout task_ids stay identical
HOLDOUT_FRACTION = 0.2

SYSTEM_MESSAGE = (
    "You are a code reviewer. Follow this exact structure: "
    "1) Briefly validate what works (1-2 sentences), "
    "2) Use 'However' to transition to specific failures, "
    "3) Reference specific test cases (Test 3, Test 5, etc.), "
    "4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."
)

all_pairs = []
with open(PREFS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        all_pairs.append(json.loads(line))

print(f"Loaded {len(all_pairs)} total preference pairs")

# --- Reuse the EXACT SAME split as SFT (seed=10) so holdout stays untouched likewise in SFT dataset alteration---
unique_task_ids = sorted(set(p["task_id"] for p in all_pairs))
random.seed(SEED)
shuffled = unique_task_ids.copy()
random.shuffle(shuffled)
num_holdout = max(1, round(len(shuffled) * HOLDOUT_FRACTION))
holdout_task_ids = set(shuffled[:num_holdout])
train_task_ids = set(shuffled[num_holdout:])

print(f"Holdout task_ids (unchanged from SFT): {sorted(holdout_task_ids)}")

# --- Build DPO records: prompt (with system msg), chosen, rejected ---
def build_dpo_record(pair):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": pair["prompt"]},
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": formatted_prompt,
        "chosen": pair["chosen"],
        "rejected": pair["rejected"],
        "task_id": pair["task_id"],
        "tool": pair["tool"],
        "tier": pair["tier"],
    }

dpo_train_records = [build_dpo_record(p) for p in all_pairs if p["task_id"] in train_task_ids]

print(f"\nDPO train examples: {len(dpo_train_records)}")

with open(DPO_TRAIN_PATH, "w", encoding="utf-8") as f:
    for rec in dpo_train_records:
        f.write(json.dumps(rec) + "\n")

print(f"Saved to {DPO_TRAIN_PATH}")
print(f"\nExample record:\n{json.dumps(dpo_train_records[0], indent=2)[:800]}")

Loaded 112 total preference pairs
Holdout task_ids (unchanged from SFT): ['task_05', 'task_08', 'task_10', 'task_18', 'task_21', 'task_23']

DPO train examples: 92
Saved to /kaggle/working/dpo_train_nosysPrmpt.jsonl

Example record:
{
  "prompt": "<|im_start|>system\nYou are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations.<|im_end|>\n<|im_start|>user\nReview this code:\n\ndef sum_numeric_strings(numbers):\n    return sum(float(n) for n in numbers)<|im_end|>\n<|im_start|>assistant\n",
  "chosen": "The function correctly uses a generator expression with `sum()` and `float()` for a clean, Pythonic approach to numeric conversion. However, it lacks defensive programming for two critical edge cases: (1) the function crashes when `numbers` is `None` instead

In [3]:
print(f"\nExample record:\n{json.dumps(dpo_train_records[0], indent=2)[:]}")


Example record:
{
  "prompt": "<|im_start|>system\nYou are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations.<|im_end|>\n<|im_start|>user\nReview this code:\n\ndef sum_numeric_strings(numbers):\n    return sum(float(n) for n in numbers)<|im_end|>\n<|im_start|>assistant\n",
  "chosen": "The function correctly uses a generator expression with `sum()` and `float()` for a clean, Pythonic approach to numeric conversion. However, it lacks defensive programming for two critical edge cases: (1) the function crashes when `numbers` is `None` instead of handling it gracefully\u2014add a guard clause like `if numbers is None: return 0` at the start; (2) it fails on malformed input like 'abc' without validation\u2014wrap the `float(n)` conversion in a try-except b

In [4]:
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

dpo_records = []
with open(DPO_TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        dpo_records.append(json.loads(line))

dpo_dataset = Dataset.from_list(dpo_records)
print(dpo_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'task_id', 'tool', 'tier'],
    num_rows: 92
})


In [5]:
print(dpo_dataset[0])

{'prompt': "<|im_start|>system\nYou are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations.<|im_end|>\n<|im_start|>user\nReview this code:\n\ndef sum_numeric_strings(numbers):\n    return sum(float(n) for n in numbers)<|im_end|>\n<|im_start|>assistant\n", 'chosen': "The function correctly uses a generator expression with `sum()` and `float()` for a clean, Pythonic approach to numeric conversion. However, it lacks defensive programming for two critical edge cases: (1) the function crashes when `numbers` is `None` instead of handling it gracefully—add a guard clause like `if numbers is None: return 0` at the start; (2) it fails on malformed input like 'abc' without validation—wrap the `float(n)` conversion in a try-except block to either skip invalid entr

In [6]:
dpo_config = DPOConfig(
    output_dir="/kaggle/working/aicr_dpo_nosysPrmpt__checkpoint",
    per_device_train_batch_size=1,       # DPO needs 2x memory (chosen+rejected per example)
    gradient_accumulation_steps=8,       # effective batch size = 8
    num_train_epochs=2,                  # DPO typically needs FEWER epochs than SFT
    learning_rate=5e-6,                  # much lower than SFT's 2e-4 — DPO is sensitive
    beta=0.1,                            # controls how strongly to diverge from the SFT reference model
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    max_length=1024,
)

trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    # no ref_model needed — PEFT automatically uses the base model (adapter disabled) as reference
)

trainer.train()



DPO_ADAPTER_PATH = "/kaggle/working/aicr_dpo_adapter-v2"

trainer.model.save_pretrained(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)

import shutil
shutil.make_archive("/kaggle/working/aicr_dpo_adapter_v2", "zip", DPO_ADAPTER_PATH)
print("Saved and zipped — download this now before doing anything else.")

Adding EOS to train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.492426
10,0.237854
15,0.157502
20,0.135015


Saved and zipped — download this now before doing anything else.


In [8]:
import json
import re
import torch
from pathlib import Path
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/input/datasets/ibtihussain/aicr-sft-adapter-v3-nosysprmpt"   # without-sysprompt SFT checkpoint
DPO_ADAPTER_PATH = "/kaggle/working/aicr_dpo_adapter-v2"       # DPO-v2 checkpoint (continued from SFT-nosysPrompt above)
HOLDOUT_PATH = "/kaggle/input/datasets/ibtihussain/test-set/sft_holdout.jsonl"
GROUND_TRUTH_PATH = "/kaggle/input/datasets/ibtihussain/ground-truth/ground_truth.json"
OUTPUT_TXT_PATH = "/kaggle/working/final_comparison_base_sft-nosysPrmpt_dpo.txt"

# System message is now HARDCODED as required — never optional at inference
SYSTEM_MESSAGE = (
    "You are a code reviewer. Follow this exact structure: "
    "1) Briefly validate what works (1-2 sentences), "
    "2) Use 'However' to transition to specific failures, "
    "3) Reference specific test cases (Test 3, Test 5, etc.), "
    "4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ============================================================
# LOAD ALL THREE MODELS — each with its own fresh base model instance
# ============================================================
def load_fresh_base():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16,
    )

print("Loading base model (no adapter)...")
base_model = load_fresh_base()
base_model.eval()
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

print("Loading SFT model...")
sft_model = PeftModel.from_pretrained(load_fresh_base(), SFT_ADAPTER_PATH)
sft_model.eval()
sft_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)

print("Loading DPO model...")
dpo_model = PeftModel.from_pretrained(load_fresh_base(), DPO_ADAPTER_PATH)
dpo_model.eval()
dpo_tokenizer = AutoTokenizer.from_pretrained(DPO_ADAPTER_PATH)

print("\nAll three models loaded fresh and independently.")

# ============================================================
# LOAD HOLDOUT SET + GROUND TRUTH
# ============================================================
holdout_raw = []
with open(HOLDOUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        holdout_raw.append(json.loads(line))

seen = set()
holdout_unique = []
for ex in holdout_raw:
    key = (ex["task_id"], ex["tool"])
    if key not in seen:
        holdout_unique.append(ex)
        seen.add(key)

print(f"Loaded {len(holdout_raw)} holdout rows, {len(holdout_unique)} unique (task_id, tool) examples")

with open(GROUND_TRUTH_PATH, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

# ============================================================
# GENERATION (system message ALWAYS included, per the lesson learned)
# ============================================================
def generate_review(model, tokenizer, prompt_text, max_new_tokens=400):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt_text},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# ============================================================
# HEURISTIC SCORING (same as before)
# ============================================================
def score_structure(response_text):
    text_lower = response_text.lower()
    has_transition = bool(re.search(r"\bhowever\b|\bbut\b", text_lower))
    has_fix_language = bool(re.search(
        r"\badd\b|\bconsider\b|\bguard clause\b|\bshould\b|\bfix\b|\bvalidat", text_lower
    ))
    avoids_walkthrough = not text_lower.strip().startswith(("###", "1.", "1)", "- **"))
    score = sum([has_transition, has_fix_language, avoids_walkthrough])
    return {"has_transition": has_transition, "has_fix_language": has_fix_language,
            "avoids_walkthrough_format": avoids_walkthrough, "structure_score_0_to_3": score}

def extract_key_terms(notes_text):
    notes_lower = notes_text.lower()
    candidates = ["none", "empty", "negative", "zero", "malformed", "timezone", "utc",
                  "version", "segment", "space", "error", "keyword", "future", "date",
                  "type", "length", "boundary", "index", "duplicate"]
    return [t for t in candidates if t in notes_lower]

def score_grounding(response_text, ground_truth_notes):
    key_terms = extract_key_terms(ground_truth_notes)
    if not key_terms:
        return {"key_terms_expected": [], "key_terms_found": [], "grounding_score": None}
    response_lower = response_text.lower()
    found = [t for t in key_terms if t in response_lower]
    return {"key_terms_expected": key_terms, "key_terms_found": found,
            "grounding_score": round(len(found) / len(key_terms), 2)}

def flag_false_negative(response_text, label):
    """For buggy snippets (edge_case_fail/hard_fail), did the model wrongly say 'no issues'?"""
    if label not in ("edge_case_fail", "hard_fail"):
        return None
    text_lower = response_text.lower()
    claims_clean = any(kw in text_lower for kw in
                        ["no issues", "clean, readable", "well-structured", "looks good",
                         "correctly handles", "well-designed", "overall, the code is clean"])
    claims_failure = bool(re.search(r"\bhowever\b|fails|crashes|test \d", text_lower))
    return {"false_negative_risk": claims_clean and not claims_failure}

# ============================================================
# RUN ALL THREE MODELS ON EVERY HELD-OUT EXAMPLE
# ============================================================
output_lines = []
output_lines.append("AICR Final Comparison — Base vs SFT vs DPO")
output_lines.append(f"Generated: {datetime.now().isoformat()}")
output_lines.append(f"System message: ALWAYS INCLUDED (hardcoded requirement)")
output_lines.append(f"Total unique held-out examples: {len(holdout_unique)}")
output_lines.append("=" * 80)

summary = {"base": {"structure": [], "grounding": [], "false_neg": 0, "false_neg_total": 0},
           "sft":  {"structure": [], "grounding": [], "false_neg": 0, "false_neg_total": 0},
           "dpo":  {"structure": [], "grounding": [], "false_neg": 0, "false_neg_total": 0}}

models = {"base": (base_model, base_tokenizer), "sft": (sft_model, sft_tokenizer), "dpo": (dpo_model, dpo_tokenizer)}

for i, ex in enumerate(holdout_unique):
    task_id, tool = ex["task_id"], ex["tool"]
    prompt_text = ex["messages"][0]["content"]
    reference_chosen = ex["messages"][1]["content"]
    gt_notes = ground_truth.get(task_id, {}).get(tool, {}).get("notes", "")
    gt_label = ground_truth.get(task_id, {}).get(tool, {}).get("label", "")

    output_lines.append(f"\n{'='*80}")
    output_lines.append(f"EXAMPLE {i+1} — task_id: {task_id}, tool: {tool}, label: {gt_label}")
    output_lines.append(f"{'='*80}")
    output_lines.append(f"\n--- PROMPT ---\n{prompt_text[:500]}")
    output_lines.append(f"\n--- GROUND TRUTH NOTES ---\n{gt_notes}")
    output_lines.append(f"\n--- REFERENCE CHOSEN ---\n{reference_chosen}")

    print(f"\n[{i+1}/{len(holdout_unique)}] {task_id}/{tool}")

    for model_key, (model, tok) in models.items():
        out = generate_review(model, tok, prompt_text)
        structure = score_structure(out)
        grounding = score_grounding(out, gt_notes)
        false_neg = flag_false_negative(out, gt_label)

        summary[model_key]["structure"].append(structure["structure_score_0_to_3"])
        if grounding["grounding_score"] is not None:
            summary[model_key]["grounding"].append(grounding["grounding_score"])
        if false_neg is not None:
            summary[model_key]["false_neg_total"] += 1
            if false_neg["false_negative_risk"]:
                summary[model_key]["false_neg"] += 1

        output_lines.append(f"\n--- {model_key.upper()} OUTPUT ---\n{out}")
        output_lines.append(f"Structure: {structure} | Grounding: {grounding} | FalseNeg: {false_neg}")

        print(f"  {model_key}: structure={structure['structure_score_0_to_3']}/3, "
              f"grounding={grounding['grounding_score']}, false_neg_risk={false_neg}")

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================
output_lines.append(f"\n{'='*80}\nFINAL SUMMARY\n{'='*80}")
print(f"\n{'='*80}\nFINAL SUMMARY\n{'='*80}")

header = f"{'Model':<8} {'Avg Structure (0-3)':<22} {'Avg Grounding (0-1)':<22} {'False Neg Rate':<18}"
output_lines.append(header)
print(header)

for key in ("base", "sft", "dpo"):
    s = summary[key]
    avg_struct = sum(s["structure"]) / len(s["structure"]) if s["structure"] else 0
    avg_ground = sum(s["grounding"]) / len(s["grounding"]) if s["grounding"] else 0
    fn_rate = s["false_neg"] / s["false_neg_total"] if s["false_neg_total"] else 0
    line = f"{key:<8} {avg_struct:<22.2f} {avg_ground:<22.2f} {fn_rate:<18.2f} ({s['false_neg']}/{s['false_neg_total']})"
    output_lines.append(line)
    print(line)

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved full report to {OUTPUT_TXT_PATH}")

Loading base model (no adapter)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading DPO model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



All three models loaded fresh and independently.
Loaded 20 holdout rows, 10 unique (task_id, tool) examples

[1/10] task_05/gpt
  base: structure=2/3, grounding=0.0, false_neg_risk={'false_negative_risk': False}
  sft: structure=2/3, grounding=1.0, false_neg_risk={'false_negative_risk': False}
  dpo: structure=2/3, grounding=1.0, false_neg_risk={'false_negative_risk': False}

[2/10] task_08/gpt
  base: structure=2/3, grounding=0.33, false_neg_risk={'false_negative_risk': False}
  sft: structure=2/3, grounding=0.33, false_neg_risk={'false_negative_risk': False}
  dpo: structure=2/3, grounding=0.67, false_neg_risk={'false_negative_risk': False}

[3/10] task_08/cursor
  base: structure=2/3, grounding=0.33, false_neg_risk={'false_negative_risk': False}
  sft: structure=2/3, grounding=0.67, false_neg_risk={'false_negative_risk': False}
  dpo: structure=2/3, grounding=0.67, false_neg_risk={'false_negative_risk': False}

[4/10] task_23/gpt
  base: structure=2/3, grounding=0.0, false_neg_risk